# Altmetric on BigQuery — setup & first query

Reproducible "hello world" for the Altmetric dataset at `altmetric-endorsements.altmetric_on_gbq`, subscribed from the **Altmetrics on BigQuery** Analytics Hub listing.

The publisher applies row-level security keyed to **your user identity**, so a service-account JSON would return zero rows — we use Application Default Credentials (your gcloud login).

The next cell runs the one-time setup commands; skip it on subsequent runs. `gcloud` is provided by the Nix devshell — see `flake.nix`.

In [ ]:
import os

# Force ADC. If GOOGLE_APPLICATION_CREDENTIALS is set (in .env or the shell),
# google-auth would prefer that service-account JSON over the user's gcloud
# login — and the Altmetric Analytics Hub RLS would filter the SA to zero
# rows. Drop it before any BigQuery call.
os.environ.pop("GOOGLE_APPLICATION_CREDENTIALS", None)

# One-time auth setup. Run once per machine, then comment out or skip.
!gcloud auth application-default login
!gcloud config set project altmetric-endorsements

In [ ]:
from pathlib import Path

from google.cloud import bigquery

client = bigquery.Client()
print(f"Project: {client.project}")

## Discover your tables and columns

Run this once to confirm the actual dataset/table names and column names. The Analytics Hub "Altmetrics on BigQuery" schema isn't guaranteed to use the field names we assumed in the example query, so we list everything before running anything.

If the table is not called `research_outputs`, or columns like `publication_date` / `score` / `title` are named differently, update `queries/altmetric_top_recent_by_score.sql` accordingly before running the query cell.

In [3]:
for dataset in client.list_datasets():
    print(dataset.dataset_id)
    for table in client.list_tables(dataset.reference):
        print(f"  - {table.table_id}")

altmetric_on_gbq
  - attention_sources
  - posts
  - research_outputs


## Example query — recent high-attention papers

The query lives in [`queries/altmetric_top_recent_by_score.sql`](queries/altmetric_top_recent_by_score.sql). It returns the 25 papers published in the last 12 months with the highest Altmetric Attention Score.

We dry-run it first to see how many bytes BigQuery would scan, then execute for real.

In [ ]:
from altendor.bigquery.preflight import preflight

sql = Path("queries/altmetric_top_recent_by_score.sql").read_text()
preflight(client, sql)

If the dry run is happy, run the real query in the next cell. Post-execution `job.total_bytes_processed` reports the actual scan size.

In [ ]:
job = client.query(sql, job_config=bigquery.QueryJobConfig(use_query_cache=False))
df = job.result().to_dataframe()

scanned = job.total_bytes_processed
billed = job.total_bytes_billed
if scanned is not None:
    print(f"Scanned: {scanned / 1e6:.2f} MB  |  Billed: {(billed or 0) / 1e6:.2f} MB")
else:
    print("Post-execution scan size masked by RLS.")
df

## Next steps

- Add new queries as `.sql` files under `notebooks/queries/`, each with the header block documented in `.claude/skills/sql/SKILL.md`.
- Prefer BigQuery query parameters (`bigquery.ScalarQueryParameter`) over Python f-string interpolation for parameterised queries.
- For ad-hoc exploration, the `%%bigquery` magic (from `bigquery-magics`) works once the client above is initialised.